# `ballpushing_utils` — a guided tour

This notebook walks you through the three data objects that form the
backbone of `ballpushing_utils`:

1. **`Fly`** — a single fly in a single corridor of a single arena.
   Binds raw tracking data (SLEAP HDF5s), arena metadata, and an
   analysis `Config` into one place.
2. **`Experiment`** — one recording session (one folder on disk), i.e.
   a collection of `Fly` objects that share metadata and configuration.
3. **`Dataset`** — the pooled DataFrame view across one or more
   experiments. This is what feeds the paper figures.

Along the way we'll peek at the `Config`, the `FlyMetadata`, and the
`FlyTrackingData` helpers so you can see how the pieces fit together.

> **Read-only mode.** The notebook is designed to also be useful even
> if you don't have the raw data handy: when `BALLPUSHING_DATA_ROOT`
> isn't reachable, the data cells print a short diagnostic and skip
> ahead. The Markdown explanations still stand on their own.

## Prerequisites

```bash
pip install -e ".[dev]"
export BALLPUSHING_DATA_ROOT=/path/to/your/dataset/root
```

`BALLPUSHING_DATA_ROOT` should point at the folder that contains your
**experiment directories** (each a folder of arenas → corridors holding
a `.mp4`, the SLEAP `*fly*.h5` / `*ball*.h5` tracks, `Metadata.json`,
and `fps.npy`). If you followed the lab layout used during paper
development, this will be the parent of `BallPushing_Exploration`,
`MagnetBlock`, `F1_Tracks`, etc.

If you only have a single fly folder to play with, point
`FLY_PATH` below at it directly in the next cell.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from ballpushing_utils import (
    Config,
    Dataset,
    Experiment,
    Fly,
    data_root,
    load_dotenv,
)

# Pick up any .env in the repo root (no-op if the file is missing).
load_dotenv(Path.cwd().parent / ".env")

# Display helpers — make DataFrames readable in Jupyter.
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

DATA_ROOT = data_root()
print(f"BALLPUSHING_DATA_ROOT resolved to: {DATA_ROOT}")
print(f"Exists on this machine? {DATA_ROOT.exists()}")

## The data model in one picture

```text
  raw SLEAP HDF5s          FlyMetadata           Config (dataclass)
  ───────────────           ───────────           ──────────────────
   fly.h5    ball.h5        Genotype, arena        interaction_threshold,
   skeleton.h5   video.mp4  corridor, brain        time_range, thresholds,
                            region, FPS …          enabled_metrics, …
           │                      │                          │
           └──────── Fly(directory=…) ──────────────────────┘
                             │
                             │ .tracking_data  (FlyTrackingData)
                             │ .event_summaries (ball-pushing metrics)
                             │ .metadata
                             ▼
                        Experiment(<recording folder>)
                             │ .flies = [Fly, Fly, …]
                             │ .fps, .metadata
                             ▼
                   Dataset(source=[Experiment, …], dataset_type=…)
                             │ .data  →  tidy pandas DataFrame
                             ▼
                  figures, stats, plotting helpers
```

Every object lazily computes what it needs on first access and caches
the result. You can start from the top (a single `Fly`) or the bottom
(a pooled `Dataset` across experiments) without changing the API.

## 1 · The `Fly` object

A fly directory on disk looks like this:

```text
<experiment_folder>/
    Metadata.json            # arena-level variables (Genotype, Pretraining, …)
    fps.npy                  # frame rate (one number)
    arena1/
        corridor1/           # ← one Fly lives here
            corridor1.mp4
            corridor1_fly.h5        # SLEAP track for the fly
            corridor1_ball.h5       # SLEAP track for the ball
            corridor1_full_body.h5  # optional skeleton track
        corridor2/
        …
    arena2/
    …
```

`Fly(directory, as_individual=True)` binds those artefacts together:
SLEAP tracks → `fly.tracking_data`, JSON values → `fly.metadata`,
analysis knobs → `fly.config`.

### 1.1 · Pick a fly

Set `FLY_PATH` below to an absolute path to one corridor folder. The
default tries a well-known fly from the wild-type control set; override
it for your own data.

In [ ]:
# --- Point this at one corridor folder on your machine ---------------
FLY_PATH = Path(os.environ.get(
    "BALLPUSHING_WALKTHROUGH_FLY",
    # A plausible default under the lab layout; override above.
    DATA_ROOT
        / "Ballpushing_Exploration/Videos/230704_FeedingState_1_AM_Videos_Tracked"
        / "arena2" / "corridor5",
))

HAVE_FLY = FLY_PATH.is_dir()
if not HAVE_FLY:
    print("⚠  Fly directory not found — the data cells below will be skipped.")
    print(f"   Tried: {FLY_PATH}")
    print("   Set BALLPUSHING_WALKTHROUGH_FLY or edit the cell above to point at a")
    print("   corridor folder that contains a .mp4 + matching *_fly.h5 / *_ball.h5.")
else:
    h5s = sorted(p.name for p in FLY_PATH.glob("*.h5"))
    print(f"Using fly directory: {FLY_PATH}")
    print(f"SLEAP tracks found:  {h5s}")

### 1.2 · The `Config` — analysis knobs

`Config` is a plain `@dataclass`. Every `Fly`, `Experiment`, and
`Dataset` holds its own instance. You never *have* to touch it — the
defaults reproduce the paper settings — but changing a threshold or
enabling a different metric set is a two-line tweak.

A handful of fields that matter most:

In [ ]:
cfg = Config()

interesting = [
    "experiment_type",
    "time_range",
    "interaction_threshold",
    "significant_threshold",
    "major_event_threshold",
    "success_direction_threshold",
    "tracks_smoothing",
    "pixels_per_mm",
    "enabled_metrics",
    "debugging",
]
rows = []
for key in interesting:
    if hasattr(cfg, key):
        val = getattr(cfg, key)
        if isinstance(val, list) and val is not None and len(val) > 5:
            val_repr = f"[{len(val)} metrics]"
        else:
            val_repr = repr(val)
        rows.append({"field": key, "default": val_repr})
pd.DataFrame(rows)

To override the defaults on a single fly, pass ``custom_config``:

```python
fly = Fly(FLY_PATH, as_individual=True,
          custom_config={"experiment_type": "MagnetBlock", "time_range": (0, None)})
```

`custom_config` accepts either a dict, a JSON path, or a YAML path. It
only mutates `fly.config`, so the rest of the pipeline keeps using your
chosen values.

### 1.3 · Construct the `Fly`

`as_individual=True` means "don't load every sibling fly in the
experiment folder — just build this one". Useful for quick experiments
or inside hermetic unit tests. Drop the flag when you want the whole
experiment loaded at once (§ 2).

In [ ]:
fly = None
if HAVE_FLY:
    fly = Fly(FLY_PATH, as_individual=True)
    print(fly)
else:
    print("(skipped — no fly directory)")

### 1.4 · `fly.metadata` — genotype, arena, and more

`FlyMetadata` pulls the arena's variables out of `Metadata.json`
(shared for the experiment) and attaches them to the fly. It also
joins the `Region_map_…csv` file to translate the cryptic `Genotype`
into a readable `nickname` + `brain_region`.

Every key in `arena_metadata` is *also* exposed as a top-level
attribute (e.g. `fly.metadata.Genotype`), so scripts can pattern-match
on attributes without going through a dict.

In [ ]:
if fly is None:
    print("(skipped)")
else:
    md = fly.metadata
    display(pd.Series({
        "name": md.name,
        "arena": md.arena,
        "corridor": md.corridor,
        "video": str(md.video),
        "fps": md.fps,
        "original_size_px": md.original_size,
        "Genotype": getattr(md, "Genotype", "n/a"),
        "nickname": md.nickname,
        "brain_region": md.brain_region,
    }, name="fly.metadata"))

    print("\nFull arena_metadata dict:")
    display(pd.Series(md.arena_metadata, name=md.name))

### 1.5 · `fly.tracking_data` — SLEAP tracks in pandas form

`FlyTrackingData` is where the heavy lifting happens: it loads the
SLEAP HDF5s, (optionally) smooths them, detects which ball is the
*training* ball vs the *test* ball (F1 protocol), filters by
`config.time_range`, and caches derived quantities like chamber-exit
times and interaction events.

The most useful handles:

- `flyball_positions` — a tidy DataFrame with the smoothed fly + ball
  coordinates per frame (and a `percent_completion` column derived
  from the ball trajectory).
- `interaction_events` — dict of detected interaction windows, keyed
  by `(fly_idx, ball_idx)`, each value a list of
  `[start_frame, end_frame, displacement_px]` triplets.
- `chamber_exit_time`, `duration`, `start_x`, `start_y` — metadata
  derived from the tracks.

In [ ]:
if fly is None or fly.tracking_data is None:
    print("(skipped)")
else:
    td = fly.tracking_data
    print(f"valid_data          : {td.valid_data}")
    print(f"chamber_exit_time s : {td.chamber_exit_time}")
    print(f"duration s          : {getattr(td, 'duration', 'n/a')}")
    print(f"n interaction events: "
          f"{sum(len(v) for d in td.interaction_events.values() for v in d.values())}")
    print("\nflyball_positions.head():")
    display(fly.flyball_positions.head())

### 1.6 · `fly.event_summaries` — the ball-pushing metrics

Accessing `event_summaries` triggers `BallPushingMetrics`, which
computes the per-ball summary statistics (number of events, first
significant event time, major-event ratios, chamber times, pause
durations, skeleton-derived features, …). This is the object your
figure scripts usually pool across flies.

The return type is a nested dict keyed by `fly_<idx>_ball_<idx>`; each
value is a flat `{metric: value}` dict.

In [ ]:
if fly is None:
    print("(skipped)")
else:
    summaries = fly.event_summaries or {}
    if not summaries:
        print("No event summaries for this fly (e.g. tracking failed).")
    else:
        # One column per ball, one row per metric. .head() keeps the cell tidy.
        tidy = pd.DataFrame(summaries).T.reset_index(names="ball_key")
        display(tidy.set_index("ball_key").head(30).T)

## 2 · The `Experiment` object

An `Experiment` is one recording session on disk: a folder of arenas
containing corridors. It owns the shared `Metadata.json` / `fps.npy`
and, by default, loads every fly inside it on construction.

Construct it either directly:

```python
exp = Experiment("/path/to/<recording_folder>")
```

or via an existing `Fly`:

```python
exp = fly.experiment   # already computed, cached
```

Pass `metadata_only=True` to skip fly construction (useful when you
only need the arena variables and FPS).

In [ ]:
exp = None
if fly is not None:
    # Reuse the experiment already built for this fly — free of charge.
    exp = fly.experiment
    # If you loaded the fly with as_individual=True, exp is metadata-only.
    if not getattr(exp, "flies", None):
        print("Rebuilding the experiment with all flies (this may take a moment)…")
        exp = Experiment(fly.directory.parent.parent)
    print(exp)
    print(f"\n#flies: {len(exp.flies)}   fps: {exp.fps}")
else:
    print("(skipped)")

### 2.1 · Surveying the flies in an experiment

`exp.flies` is a plain list of `Fly` objects. Invalid flies (missing
tracks, tagged `Genotype == "none"`, failed premature-exit checks,
etc.) are filtered out by the loader, so the list is already clean.

A one-liner to see the genotype × brain-region breakdown:

In [ ]:
if exp is None or not exp.flies:
    print("(skipped)")
else:
    survey = pd.DataFrame([
        {
            "name": f.metadata.name,
            "arena": f.metadata.arena,
            "corridor": f.metadata.corridor,
            "nickname": f.metadata.nickname,
            "brain_region": f.metadata.brain_region,
            "Genotype": getattr(f.metadata, "Genotype", "?"),
        }
        for f in exp.flies
    ])
    display(survey.head(15))
    print(f"\nGenotype × brain_region counts:")
    display(survey.groupby(["nickname", "brain_region"]).size()
                  .rename("n_flies").to_frame())

`Experiment.find_flies(on, value)` gives you a filtered list:

```python
exp.find_flies("metadata.nickname", "PR")
exp.find_flies("Genotype", "w1118")
```

Nested attribute access works (dot-separated paths). Returns `None`
gracefully for missing attributes instead of raising.

## 3 · The `Dataset` object

A `Dataset` turns one or more `Experiment`s (or a list of `Fly`s) into
a **single tidy pandas DataFrame**, ready for plotting and stats. The
shape of that DataFrame is controlled by `dataset_type`:

| `dataset_type` | one row per … | main use |
| --- | --- | --- |
| `"summary"` | (fly, ball) | bar/box/permutation plots across genotypes |
| `"coordinates"` | (fly, frame) | trajectory plots, velocity profiles |
| `"fly_positions"` | (fly, frame, keypoint) | skeleton-level analyses |
| `"event_metrics"` | interaction event | per-event stats |
| `"F1_coordinates"` | (fly, frame) but F1-specific | Fig 2 heatmaps / traces |
| `"F1_checkpoints"` | F1 milestone | Fig 2 pretraining ladder |
| `"contact_data"` | skeleton-contact frame | Fig 3 contact timelines |
| `"transformed"` / `"transposed"` | event × frame features | UMAP / downstream ML |
| `"behavior_umap"` | UMAP embedding row | Behaviour UMAP panel |

The `source` argument accepts:

- a single `Fly` or `Experiment`
- a list of `Fly`s (pool across corridors you hand-picked)
- a list of `Experiment`s (pool across recording days)

In [ ]:
ds = None
if exp is None or not exp.flies:
    print("(skipped)")
else:
    # Build a summary-metrics dataset for the whole experiment.
    ds = Dataset(exp, dataset_type="summary")
    print(ds)
    print(f"\nds.data.shape = {ds.data.shape}")
    display(ds.data.head())

### 3.1 · Pooling across experiments

Pass a list of experiments to pool. The resulting DataFrame is a plain
concatenation — `_add_metadata` stamps each row with the arena / fly
metadata so you can `groupby` however you like.

```python
exp_a = Experiment(data_root() / "BallPushing_Exploration/.../day_a")
exp_b = Experiment(data_root() / "BallPushing_Exploration/.../day_b")
ds = Dataset([exp_a, exp_b], dataset_type="summary")
```

For production pipelines, pre-computed pooled datasets are typically
persisted as `.feather` files (see `figures/Fig1-setup/` for examples).
The `Dataset` class is mostly a convenience for ad-hoc analyses and
the upstream `.feather` pipeline.

### 3.2 · Filtering on the fly

`Dataset.find_flies(on, value)` mirrors the `Experiment` version, but
works across all pooled flies:

```python
controls = ds.find_flies("metadata.brain_region", "Control")
control_ds = Dataset(controls, dataset_type="summary")
```

…then pass the filtered sub-dataset to any of the shared plotting
helpers or straight to `pandas`.

## 4 · Where to go from here

Now that you have a `Fly` / `Experiment` / `Dataset` in hand, the rest
of the package fans out into:

- **`ballpushing_utils.ballpushing_metrics`** — the guts behind
  `fly.event_summaries`. Good starting point to add a new metric.
- **`ballpushing_utils.plotting`** — shared Matplotlib/Seaborn helpers
  that the paper figures call (`paired_boxplot_with_significance`,
  colour palettes, figure-output directory scaffolding).
- **`ballpushing_utils.stats`** — `permutation_test`, effect-size and
  correction helpers. Reproducible against the paper's published
  p-values (see `tests/unit/stats/`).
- **`ballpushing_utils.diagnostics`** — one-shot QA for a single fly:
  per-event timeline with video-scrubbing timestamps, and metric range
  checks against `DEFAULT_METRIC_RANGES`. See
  [`notebooks/diagnostics_demo.ipynb`](./diagnostics_demo.ipynb) for
  a hermetic walkthrough and
  `tools/diagnostics_dashboard.py` for the interactive Panel view.
- **`figures/`** — one subfolder per paper figure, each a self-contained
  script that reads a `.feather`, calls the plotting+stats helpers,
  and writes its panel to `figures_root()`. Read these to see the
  full end-to-end pattern.

### A two-line recipe for a new analysis

```python
from ballpushing_utils import Experiment, Dataset

ds = Dataset(Experiment("/path/to/recording"), dataset_type="summary")
df = ds.data.query("nickname.isin(['PR', 'MBON01'])")
```

…and you're in pandas land. Good luck!